# Example 4.3.1 / 1: Some Versus Every Member

Companion notebook for **4.3.1 exists and forall** in the AMPL lab Documentation, section "Some Versus Every Member". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Some Versus Every Member

The model asks whether any product has a negative margin and whether every product meets a
minimum margin. `ampl.get_value()` returns the two logical values.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PRODUCTS;             # products
param margin {PRODUCTS};  # profit per ton
param required_margin;    # margin threshold to test against

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PRODUCTS"] = ["bands", "coils", "plate"]
ampl.param["margin"] = {"bands": 15, "coils": -2, "plate": 8}
ampl.param["required_margin"] = 5

In [5]:
# run.py
print(ampl.get_value("if exists {p in PRODUCTS} margin[p] < 0 then 1 else 0"))
print(ampl.get_value("if forall {p in PRODUCTS} margin[p] >= required_margin then 1 else 0"))

1
0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PRODUCTS := bands coils plate;
param margin := bands 15  coils -2  plate 8;
param required_margin := 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display if exists {p in PRODUCTS} margin[p] < 0 then 1 else 0;
display if forall {p in PRODUCTS} margin[p] >= required_margin then 1 else 0;

 if exists{p in PRODUCTS} margin[p] < 0 then 1 = 1

 if forall{p in PRODUCTS} margin[p] >= required_margin then 1 = 0

